
# 🐍 Python for Data Engineering — Basic to Advanced
### A Hands-On, Runnable Notebook by Your Python Tutor

Welcome! This notebook is built specifically for **Data Engineering** — the topics, examples,
and mini-exercises all revolve around problems you'll actually face: reading messy files,
cleaning data, talking to databases and APIs, and building small ETL pipelines.

**How to use this notebook**
- Run cells top to bottom (`Shift + Enter`). Later topics build on earlier ones.
- Every topic has: **Concept explanation → Runnable example → Data-Engineering angle → Try-it-yourself exercise**.
- Exercises have hidden solutions (commented out) — try first, then peek!

**Roadmap (20 Modules)**

| # | Module | Level |
|---|--------|-------|
| 1 | Python Basics Refresher | Basic |
| 2 | Operators & Control Flow | Basic |
| 3 | Functions & Lambdas | Basic |
| 4 | Core Data Structures | Basic |
| 5 | Strings & Regular Expressions | Basic |
| 6 | Comprehensions & Generators | Intermediate |
| 7 | Exception Handling & Logging | Intermediate |
| 8 | File Handling (TXT/CSV/JSON) | Intermediate |
| 9 | Dates & Times | Intermediate |
| 10 | Object-Oriented Programming | Intermediate |
| 11 | Modules, Packages & Virtual Envs | Intermediate |
| 12 | NumPy Fundamentals | Intermediate |
| 13 | Pandas — DataFrames | Intermediate/Advanced |
| 14 | Data Cleaning & Transformation | Advanced |
| 15 | Databases with SQLite & SQLAlchemy | Advanced |
| 16 | Consuming APIs & JSON | Advanced |
| 17 | Columnar Formats — Parquet (PyArrow) | Advanced |
| 18 | Concurrency & Parallelism | Advanced |
| 19 | Config, Environment & Secrets | Advanced |
| 20 | Capstone — End-to-End Mini ETL Pipeline | Advanced |
---


In [2]:

# Environment check — run this first
import sys
print("Python version:", sys.version)
print("Ready to go! 🚀")


Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Ready to go! 🚀


In [4]:
#Install required libraries
!pip install sqlalchemy pyarrow -q


## Module 1 — Python Basics Refresher

**Concept:** Variables in Python are labels pointing to objects in memory. Python is
*dynamically typed* (a variable's type is decided at runtime) but *strongly typed*
(it won't silently convert `"5" + 5`, that's a `TypeError`).

**Core built-in types:** `int`, `float`, `str`, `bool`, `NoneType`, and the collections
`list`, `tuple`, `dict`, `set` (covered in Module 4).

**Why it matters for Data Engineering:** Every pipeline starts with raw values coming from
files, APIs, or databases — usually as strings. Knowing exactly how Python types behave
(and coerce) prevents subtle bugs like `"100" + "1"` giving `"1001"` instead of `101`.


In [7]:

# Variables & dynamic typing
record_id = 1042          # int
price = 19.99              # float
product_name = "USB Cable" # str
in_stock = True            # bool
supplier = None            # NoneType (missing value, like SQL NULL)

for var in (record_id, price, product_name, in_stock, supplier):
    print(var, "type : ", type(var).__name__ )


1042 type :  int
19.99 type :  float
USB Cable type :  str
True type :  bool
None type :  NoneType


In [9]:

# Type coercion pitfalls common when reading raw data (e.g. from CSV, all strings!)
raw_quantity = "25"          # this is what you get from a CSV row
raw_price = "3.5"

# WRONG: string concatenation, not addition
print("Buggy total (string concat):", raw_quantity + raw_price)

# RIGHT: explicit casting before doing math
quantity = int(raw_quantity)
price = float(raw_price)
total = quantity * price
print("Correct total:", total)

# f-strings are the standard for readable, fast string formatting
print(f"Order summary: {quantity} units @ ${price:.2f} = ${total:.2f}")


Buggy total (string concat): 253.5
Correct total: 87.5
Order summary: 25 units @ $3.50 = $87.50


In [12]:
raw = "007"
raw_id = int(raw)
raw_salary = "45000.51234"
raw_salary_f = float(raw_salary)
adjusted_salary = raw_salary_f * 1.1
print("salary", adjusted_salary)

print(f"Employee ID {raw_id}: salary = {raw_salary_f * 1.1:.2f}")


salary 49500.56357400001
Employee ID 7: salary = 49500.56



**Try it yourself:** You receive `raw = "007"` (a zero-padded employee ID) and
`raw_salary = "45000.5"`. Convert both to appropriate numeric types and print
`salary * 1.1` (a 10% raise) formatted to 2 decimal places.


In [ ]:

# Your code here


# --- Solution (uncomment to check) ---
# raw = "007"
# raw_salary = "45000.5"
# emp_id = int(raw)
# salary = float(raw_salary)
# print(f"Employee {emp_id}: new salary = {salary * 1.1:.2f}")



## Module 2 — Operators & Control Flow

**Concept:** Control flow (`if/elif/else`, `for`, `while`) determines which code runs
and how many times. Python has no `switch` statement (pre-3.10) but 3.10+ has
`match/case` for structural pattern matching — handy for routing records by type.

**Data-Engineering angle:** Pipelines constantly branch on data quality — "if the value
is null, skip; if it's malformed, log and quarantine; otherwise process." Loops drive
row-by-row or batch processing.


In [32]:

# Comparison & logical operators used constantly in data validation
row = {"id": 5, "amount": -12.5, "status": "PENDING"}

is_valid_amount = float(row["amount"]) > 0
is_known_status = row["status"] in {"PENDING", "COMPLETE", "FAILED"}

if is_valid_amount and is_known_status:
    print("Row is clean ✅")
elif not is_valid_amount and is_known_status:
    print("Row flagged: negative amount ⚠️")
else:
    print("Row rejected: unknown status ❌")


Row flagged: negative amount ⚠️


In [35]:

# match/case (Python 3.10+) — great for routing records by type/status
def route_record(status: str) -> str:
    match status:
        case "PENDING":
            return "queue_for_processing"
        case "COMPLETE" | "SUCCESS":
            return "archive"
        case "FAILED" | "ERROR":
            return "dead_letter_queue"
        case _:
            return "unknown_handler"

for s in ["PENDING", "SUCCESS", "ERROR", "WEIRD_STATUS", " 123"]:
    print(f"{s:>15} -> {route_record(s)}")


        PENDING -> queue_for_processing
        SUCCESS -> archive
          ERROR -> dead_letter_queue
   WEIRD_STATUS -> unknown_handler
            123 -> unknown_handler


In [41]:

# for / while loops — simulating batch processing with early exit (break) and skip (continue)
batch = [10, 20, -5, 0, 42, None, 7]
total = 0
processed = 0

for value in batch:
    if value is None:
        continue                      # skip missing values
    if value < 0:
        print(f"Stopping: negative value {value} indicates corrupt batch")
        break                         # stop processing on bad data
    total += value
    processed += 1

print(f"Processed {processed} records, running total = {total}")


Stopping: negative value -5 indicates corrupt batch
Processed 2 records, running total = 30



**Try it yourself:** Given `statuses = ["OK","OK","TIMEOUT","OK","TIMEOUT","TIMEOUT"]`,
loop through and count consecutive `"TIMEOUT"`s. If you see 3 in a row, print
`"Circuit breaker triggered!"` and stop (this is a real pattern used in ingestion
pipelines to stop hammering a failing API).


In [42]:

# Your code here


# --- Solution ---
# statuses = ["OK","OK","TIMEOUT","OK","TIMEOUT","TIMEOUT"]
# consecutive = 0
# for s in statuses:
#     if s == "TIMEOUT":
#         consecutive += 1
#         if consecutive == 3:
#             print("Circuit breaker triggered!")
#             break
#     else:
#         consecutive = 0



## Module 3 — Functions & Lambda Expressions

**Concept:** Functions package reusable logic. Python supports default arguments,
`*args`/`**kwargs` for flexible signatures, type hints for clarity, and `lambda` for
small anonymous functions (great as arguments to `sorted()`, `map()`, `filter()`).

**Data-Engineering angle:** Transformation logic (cleaning a field, computing a derived
column, validating a record) should live in small, testable, well-named functions —
this is the backbone of maintainable pipelines.


In [26]:

from typing import Optional

def clean_price(raw_value: str, default: Optional[float] = None) -> Optional[float]:
    '''Convert a raw price string to float, stripping currency symbols/commas.
    Returns `default` if conversion fails (bad/missing data).
    '''
    if raw_value is None:
        #print(f"raw_value: {raw_value}")
        return default

    cleaned = raw_value.replace("$", "").replace(",", "").strip()
    try:
        #print(f"cleaned_value: {cleaned}")
        return float(cleaned)
    except ValueError:
        #print(f"Error: {ValueError}")
        return default

samples = ["$1,200.50", "45.0", "N/A", None, "  99.99 "]

for s in samples:
    print(f"{clean_price(s, default=0.0)}")


1200.5
45.0
0.0
0.0
99.99


In [65]:

# *args / **kwargs — flexible function signatures, common in generic pipeline utilities
def build_record(record_id: int, *tags: str, **fields) -> dict:
    '''Build a record dict from a required id, optional tags, and arbitrary key=value fields.'''
    return {"id": record_id, "tags": list(tags), **fields}

records = []
records.append(build_record(101, "priority", "eu-region", customer="Acme Corp", amount=500))
records.append(build_record(102, "priority2", "eu-region2", customer="Acme Corp2", amount=2500))
print(records)


[{'id': 101, 'tags': ['priority', 'eu-region'], 'customer': 'Acme Corp', 'amount': 500}, {'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2500}]


In [46]:
#Iterate Through Records
for record in records:
    print(record)

{'id': 101, 'tags': ['priority', 'eu-region'], 'customer': 'Acme Corp', 'amount': 500}
{'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2500}


In [47]:
#access particular column
for record in records:
    print(record["customer"])

Acme Corp
Acme Corp2


In [50]:
#filter records
filtered = []

for record in records:
    if record["amount"] > 1000:
        filtered.append(record)

print(filtered)

[{'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2500}]


In [51]:
#list comprehensive

filtered = [
    record
    for record in records
    if record["amount"] > 1000
]

print(filtered)

[{'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2500}]


In [52]:
#Calculate Total Amount
total = 0

for record in records:
    total += record["amount"]

print(total)

3000


In [53]:
# using sum
total = sum(record["amount"] for record in records)

print(total)

3000


In [54]:
#avg of records
avg = sum(r["amount"] for r in records) / len(records)

print(avg)

1500.0


In [55]:
#highest record
highest = max(records, key=lambda r: r["amount"])

print(highest)

{'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2500}


In [56]:
#minimum record
lowest = min(records, key=lambda r: r["amount"])

print(lowest)

{'id': 101, 'tags': ['priority', 'eu-region'], 'customer': 'Acme Corp', 'amount': 500}


In [57]:
# sort records

sorted_records = sorted(
    records,
    key=lambda r: r["amount"]
)

print(sorted_records)

sorted_records = sorted(
    records,
    key=lambda r: r["amount"],
    reverse=True
)

print(sorted_records)

[{'id': 101, 'tags': ['priority', 'eu-region'], 'customer': 'Acme Corp', 'amount': 500}, {'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2500}]
[{'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2500}, {'id': 101, 'tags': ['priority', 'eu-region'], 'customer': 'Acme Corp', 'amount': 500}]


In [58]:
# update records

for record in records:
    record["amount"] *= 1.10

print(records)

[{'id': 101, 'tags': ['priority', 'eu-region'], 'customer': 'Acme Corp', 'amount': 550.0}, {'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2750.0}]


In [67]:
#Add new column
for record in records:
    record["currency"] = "USD"

print(records)

[{'id': 101, 'tags': ['priority', 'eu-region'], 'customer': 'Acme Corp', 'amount': 500, 'currency': 'USD'}, {'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2500, 'currency': 'USD'}]


In [ ]:
#Rename column
for record in records:
    record["company"] = record.pop("customer")

print(records)

In [68]:
#delete column
for record in records:
    del record["currency"]

print(records)

[{'id': 101, 'tags': ['priority', 'eu-region'], 'customer': 'Acme Corp', 'amount': 500}, {'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2500}]


In [ ]:
#find specific record
for record in records:
    if record["id"] == 102:
        print(record)

In [61]:
#Convert into Dictionary
lookup = {
    record["id"]: record
    for record in records
}

print(lookup)

{101: {'id': 101, 'customer': 'Acme Corp', 'amount': 550.0, 'currency': 'USD'}, 102: {'id': 102, 'customer': 'Acme Corp2', 'amount': 2750.0, 'currency': 'USD'}}


In [62]:
# count records
len(records)

2

In [69]:
#Group By (Very Common)

records.append(build_record(
    103,
    "priority",
    "us-region",
    customer="ABC",
    amount=700
))

records.append(build_record(
    104,
    "priority",
    "eu-region",
    customer="XYZ",
    amount=900
))

groups = {}

for record in records:
    tag = record["tags"][0]

    groups.setdefault(tag, []).append(record)

print(groups)

{'priority': [{'id': 101, 'tags': ['priority', 'eu-region'], 'customer': 'Acme Corp', 'amount': 500}, {'id': 103, 'tags': ['priority', 'us-region'], 'customer': 'ABC', 'amount': 700}, {'id': 104, 'tags': ['priority', 'eu-region'], 'customer': 'XYZ', 'amount': 900}], 'priority2': [{'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2500}]}


In [70]:
#Count by Group
counts = {}

for record in records:
    tag = record["tags"][0]

    counts[tag] = counts.get(tag, 0) + 1

print(counts)

{'priority': 3, 'priority2': 1}


In [73]:
#unique records - remove duplicate
unique = {}

for record in records:
    unique[record["id"]] = record

records = list(unique.values())

print(records)

[{'id': 101, 'tags': ['priority', 'eu-region'], 'customer': 'Acme Corp', 'amount': 500}, {'id': 102, 'tags': ['priority2', 'eu-region2'], 'customer': 'Acme Corp2', 'amount': 2500}, {'id': 103, 'tags': ['priority', 'us-region'], 'customer': 'ABC', 'amount': 700}, {'id': 104, 'tags': ['priority', 'eu-region'], 'customer': 'XYZ', 'amount': 900}]


In [74]:
#Extract Single Column

customers = [
    record["customer"]
    for record in records
]

print(customers)

['Acme Corp', 'Acme Corp2', 'ABC', 'XYZ']


In [75]:
#Convert to Pandas DataFrame

import pandas as pd

df = pd.DataFrame(records)

print(df)

    id                     tags    customer  amount
0  101    [priority, eu-region]   Acme Corp     500
1  102  [priority2, eu-region2]  Acme Corp2    2500
2  103    [priority, us-region]         ABC     700
3  104    [priority, eu-region]         XYZ     900


In [76]:

# Lambdas — short, throwaway functions, most useful with sorted/map/filter
transactions = [
    {"id": 1, "amount": 250.0, "type": "debit"},
    {"id": 2, "amount": 75.5,  "type": "credit"},
    {"id": 3, "amount": 900.0, "type": "debit"},
]

# Sort by amount descending
by_amount = sorted(transactions, key=lambda t: t["amount"], reverse=True)
print("Sorted by amount:", [t["id"] for t in by_amount])

# Filter only debits
debits = list(filter(lambda t: t["type"] == "debit", transactions))
print("Debits:", [t["id"] for t in debits])

# Map: extract just the amounts
amounts = list(map(lambda t: t["amount"], transactions))
print("Amounts:", amounts)


Sorted by amount: [3, 1, 2]
Debits: [1, 3]
Amounts: [250.0, 75.5, 900.0]



**Try it yourself:** Write a function `is_valid_email(addr: str) -> bool` that returns
`True` if the string contains exactly one `"@"` and at least one `"."` after it. Test it
against `["a@b.com", "no-at-symbol", "no-at.@.at.symbol", "two@@at.com", "user@sitecom"]`.


In [88]:

# Your code here


# --- Solution ---
def is_valid_email(addr: str) -> bool:
    if addr.count("@") != 1:
        return False

    local, domain = addr.split("@")

    if local == "" or domain == "":
        return False

    if "." not in domain:
        return False

    if domain.startswith(".") or domain.endswith("."):
        return False

    return True

for e in ["a@b.com", "no-at-symbol", "no-at.@.at.symbol", "two@@at.com", "user@sitecom"]:
    print(e, "->", is_valid_email(e))


a@b.com -> True
no-at-symbol -> False
no-at.@.at.symbol -> False
two@@at.com -> False
user@sitecom -> False



## Module 4 — Core Data Structures for Data Engineering

**Concept:** Python's four built-in collections each have a distinct purpose:
- `list` — ordered, mutable, allows duplicates → sequences of records
- `tuple` — ordered, **immutable** → fixed-shape rows, dict keys, function returns
- `dict` — key→value mapping → JSON-like records, lookups by ID
- `set` — unordered, unique elements → deduplication, membership tests (O(1) average)

**Data-Engineering angle:** A "row" is usually a `dict` or a `tuple`. A "table" is a
`list` of those. Deduplicating IDs? Use a `set`. These choices affect both correctness
and performance at scale.


In [ ]:

# List: ordered collection of records
orders = [
    {"id": 1, "customer": "Alice", "amount": 250.0},
    {"id": 2, "customer": "Bob",   "amount": 75.5},
    {"id": 3, "customer": "Alice", "amount": 900.0},
]

# Tuple: immutable — great for fixed schema rows or composite keys
composite_key = (2024, "APAC", "electronics")  # (year, region, category)
print("Composite key:", composite_key)
# composite_key[0] = 2025   # <-- would raise TypeError: tuples are immutable

# Dict: fast key-based lookup — build an index from id -> record
orders_by_id = {o["id"]: o for o in orders}
print("Lookup order 2:", orders_by_id[2])

# Set: uniqueness & fast membership checks
customers_seen = {o["customer"] for o in orders}
print("Unique customers:", customers_seen)
print("Is 'Alice' a customer?", "Alice" in customers_seen)   # O(1) average


Composite key: (2024, 'APAC', 'electronics')
Lookup order 2: {'id': 2, 'customer': 'Bob', 'amount': 75.5}
Unique customers: {'Alice', 'Bob'}
Is 'Alice' a customer? True


In [ ]:

# Common data-engineering patterns with dict/list

# 1) Grouping (manual groupby) — total amount per customer
totals = {}
for o in orders:
    totals[o["customer"]] = totals.get(o["customer"], 0) + o["amount"]
print("Totals per customer:", totals)

# 2) collections.Counter — frequency counts (e.g. status codes in a log)
from collections import Counter
status_log = ["200", "200", "404", "500", "200", "404"]
print("Status counts:", Counter(status_log))

# 3) collections.defaultdict — avoid KeyError boilerplate when grouping
from collections import defaultdict
by_customer = defaultdict(list)
for o in orders:
    by_customer[o["customer"]].append(o["id"])
print("Order IDs per customer:", dict(by_customer))


Totals per customer: {'Alice': 1150.0, 'Bob': 75.5}
Status counts: Counter({'200': 3, '404': 2, '500': 1})
Order IDs per customer: {'Alice': [1, 3], 'Bob': [2]}



**Try it yourself:** You have `events = [("u1","login"), ("u2","login"), ("u1","click"),
("u1","login"), ("u2","logout")]`. Build a `dict` mapping each user to the **set** of
unique event types they triggered.


In [ ]:

# Your code here


# --- Solution ---
# events = [("u1","login"), ("u2","login"), ("u1","click"), ("u1","login"), ("u2","logout")]
# from collections import defaultdict
# user_events = defaultdict(set)
# for user, event in events:
#     user_events[user].add(event)
# print(dict(user_events))



## Module 5 — String Manipulation & Regular Expressions

**Concept:** Raw data is messy text. Core string methods (`.strip()`, `.split()`,
`.replace()`, `.lower()`) handle simple cleaning. For pattern-based extraction/validation
(emails, phone numbers, log lines), use the `re` module.

**Data-Engineering angle:** Parsing log files, extracting fields from semi-structured
text, and validating formats (dates, IDs, emails) are everyday tasks in ingestion
pipelines.


In [ ]:

# Basic string cleaning — a very common "normalize this messy column" task
raw_names = ["  ALICE johnson ", "bob-SMITH", "Carla   Diaz"]

def normalize_name(name: str) -> str:
    name = name.strip()                 # remove leading/trailing whitespace
    name = name.replace("-", " ")       # normalize separators
    name = " ".join(name.split())       # collapse multiple spaces
    return name.title()                 # Title Case

for n in raw_names:
    print(f"{n!r:>20} -> {normalize_name(n)!r}")


  '  ALICE johnson ' -> 'Alice Johnson'
         'bob-SMITH' -> 'Bob Smith'
      'Carla   Diaz' -> 'Carla Diaz'


In [ ]:

import re

# Regex: extracting structured fields from a semi-structured log line
log_line = '2026-07-08 14:32:11 [ERROR] user_id=4471 msg="Payment failed" code=402'

pattern = r'(?P<date>[\d-]+) (?P<time>[\d:]+) \[(?P<level>\w+)\] user_id=(?P<user_id>\d+) msg="(?P<msg>[^"]+)" code=(?P<code>\d+)'
match = re.search(pattern, log_line)

if match:
    parsed = match.groupdict()
    for key, val in parsed.items():
        print(f"{key:>10}: {val}")


      date: 2026-07-08
      time: 14:32:11
     level: ERROR
   user_id: 4471
       msg: Payment failed
      code: 402


In [ ]:

# Regex for validation — checking a batch of raw phone numbers
phones = ["+1-555-234-1234", "555.234.1234", "not-a-phone", "5552341234"]
phone_pattern = re.compile(r'^\+?1?[-.\s]?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}$')

for p in phones:
    status = "valid ✅" if phone_pattern.match(p) else "invalid ❌"
    print(f"{p:>18} -> {status}")

# re.sub — bulk cleaning: strip all non-digit characters
digits_only = [re.sub(r'\D', '', p) for p in phones]
print("Digits only:", digits_only)


   +1-555-234-1234 -> valid ✅
      555.234.1234 -> valid ✅
       not-a-phone -> invalid ❌
        5552341234 -> valid ✅
Digits only: ['15552341234', '5552341234', '', '5552341234']



**Try it yourself:** Given `messy = "Revenue:  $1,204.50 | Cost:$980.10|Profit:$224.40"`,
use `re.findall` with a pattern to extract all three dollar amounts as strings
(e.g. `['1,204.50', '980.10', '224.40']`).


In [ ]:

# Your code here


# --- Solution ---
# messy = "Revenue:  $1,204.50 | Cost:$980.10|Profit:$224.40"
# amounts = re.findall(r'\$([\d,]+\.\d{2})', messy)
# print(amounts)



## Module 6 — Comprehensions & Generators (Memory-Efficient Processing)

**Concept:** List/dict/set comprehensions are concise, Pythonic ways to build
collections. **Generators** (`yield`, or generator expressions `(x for x in ...)`)
produce values *lazily*, one at a time, instead of building the whole collection in
memory.

**Data-Engineering angle:** This is a *big deal* at scale. Reading a 10 GB CSV into a
`list` will blow up your memory; a generator processes it row-by-row with a small,
constant memory footprint. This is the foundation of streaming ETL.


In [ ]:

# List / dict / set comprehensions
numbers = range(1, 11)

squares = [n**2 for n in numbers if n % 2 == 0]           # list comprehension
square_map = {n: n**2 for n in numbers}                    # dict comprehension
unique_remainders = {n % 3 for n in numbers}                # set comprehension

print("Even squares:", squares)
print("Square map (first 3):", {k: square_map[k] for k in list(square_map)[:3]})
print("Unique remainders mod 3:", unique_remainders)


Even squares: [4, 16, 36, 64, 100]
Square map (first 3): {1: 1, 2: 4, 3: 9}
Unique remainders mod 3: {0, 1, 2}


In [ ]:

import sys

# Generators vs Lists — memory comparison
def make_list(n):
    return [i * i for i in range(n)]      # materializes ALL n items in memory

def make_generator(n):
    return (i * i for i in range(n))       # produces items lazily, on demand

n = 1_000_000
a_list = make_list(n)
a_gen = make_generator(n)

print(f"List size in memory:      {sys.getsizeof(a_list):,} bytes")
print(f"Generator size in memory: {sys.getsizeof(a_gen):,} bytes")
print("-> The generator stays tiny no matter how large `n` gets.")


List size in memory:      8,448,728 bytes
Generator size in memory: 200 bytes
-> The generator stays tiny no matter how large `n` gets.


In [ ]:

# A generator FUNCTION using `yield` — simulates streaming rows from a huge file
def read_large_file_simulated(total_rows: int):
    '''Pretend this reads a huge file line by line instead of loading it all.'''
    for i in range(total_rows):
        yield {"row_id": i, "value": i * 1.5}

# We never hold more than ONE row in memory at a time
running_total = 0.0
count = 0
for row in read_large_file_simulated(100_000):
    running_total += row["value"]
    count += 1

print(f"Streamed {count:,} rows, total value = {running_total:,.1f}")


Streamed 100,000 rows, total value = 7,499,925,000.0


**Try it yourself:** Write a generator function `batch(iterable, size)` that yields
successive chunks (as lists) of the given `size` from `iterable`. This is exactly the
pattern used to send data to an API or DB in manageable batches. Test it with
`list(batch(range(10), 3))` → should give `[[0,1,2],[3,4,5],[6,7,8],[9]]`.


In [ ]:

# Your code here


# --- Solution ---
# def batch(iterable, size):
#     chunk = []
#     for item in iterable:
#         chunk.append(item)
#         if len(chunk) == size:
#             yield chunk
#             chunk = []
#     if chunk:
#         yield chunk
#
# print(list(batch(range(10), 3)))



## Module 7 — Exception Handling & Logging

**Concept:** `try/except/else/finally` lets you handle errors gracefully instead of
crashing. Catch **specific** exceptions (not a bare `except:`), and use `logging`
instead of `print()` for anything that runs in production.

**Data-Engineering angle:** Pipelines run unattended. A single bad row shouldn't crash
a 6-hour job — you want to catch it, log it with context, route it to a "dead letter"
location, and keep going.


In [ ]:

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("etl_pipeline")

def parse_row(raw: dict) -> dict:
    '''Parses and validates one raw row, raising specific errors on bad data.'''
    if "amount" not in raw:
        raise KeyError("amount")
    try:
        amount = float(raw["amount"])
    except (TypeError, ValueError):
        raise ValueError(f"Non-numeric amount: {raw['amount']!r}")
    if amount < 0:
        raise ValueError(f"Negative amount not allowed: {amount}")
    return {"id": raw["id"], "amount": amount}

raw_rows = [
    {"id": 1, "amount": "100.50"},
    {"id": 2, "amount": "oops"},
    {"id": 3},
    {"id": 4, "amount": "-50"},
]

clean_rows, dead_letter = [], []

for raw in raw_rows:
    try:
        clean_rows.append(parse_row(raw))
    except KeyError as e:
        logger.warning(f"Row {raw.get('id')} missing field: {e}")
        dead_letter.append(raw)
    except ValueError as e:
        logger.warning(f"Row {raw.get('id')} failed validation: {e}")
        dead_letter.append(raw)
    else:
        logger.info(f"Row {raw['id']} parsed successfully")
    finally:
        pass  # e.g. metrics.increment("rows_attempted") would go here

print(f"\nClean rows: {clean_rows}")
print(f"Dead letter queue: {dead_letter}")


11:30:18 | INFO     | Row 1 parsed successfully


11:30:18 | WARNING  | Row 2 failed validation: Non-numeric amount: 'oops'


11:30:18 | WARNING  | Row 3 missing field: 'amount'


11:30:18 | WARNING  | Row 4 failed validation: Negative amount not allowed: -50.0



Clean rows: [{'id': 1, 'amount': 100.5}]
Dead letter queue: [{'id': 2, 'amount': 'oops'}, {'id': 3}, {'id': 4, 'amount': '-50'}]


In [ ]:

# Custom exceptions — give your pipeline's errors meaningful, catchable names
class DataQualityError(Exception):
    '''Raised when a record fails a data-quality rule.'''

class SchemaMismatchError(DataQualityError):
    '''Raised when a record's structure doesn't match the expected schema.'''

def validate_schema(record: dict, expected_keys: set):
    if set(record.keys()) != expected_keys:
        missing = expected_keys - record.keys()
        extra = record.keys() - expected_keys
        raise SchemaMismatchError(f"missing={missing}, extra={extra}")

try:
    validate_schema({"id": 1, "amount": 10}, expected_keys={"id", "amount", "currency"})
except SchemaMismatchError as e:
    print(f"Caught custom exception: {e}")
except DataQualityError:
    print("Caught a general data quality issue")


Caught custom exception: missing={'currency'}, extra=set()


**Try it yourself:** Write a function `safe_divide(a, b)` that returns `a / b`, but
catches `ZeroDivisionError` and returns `None` instead, logging a warning via
`logger.warning(...)`. Test with `safe_divide(10, 2)` and `safe_divide(10, 0)`.


In [ ]:

# Your code here


# --- Solution ---
# def safe_divide(a, b):
#     try:
#         return a / b
#     except ZeroDivisionError:
#         logger.warning(f"Division by zero: {a}/{b}")
#         return None
#
# print(safe_divide(10, 2))
# print(safe_divide(10, 0))



## Module 8 — File Handling (Text, CSV, JSON)

**Concept:** Always use `with open(...)` (a *context manager*) so files are closed
automatically, even on error. Python's `csv` and `json` modules handle the fiddly
parsing details (quoting, escaping) for you — don't hand-roll a CSV parser with
`.split(",")`, it breaks the moment a field contains a comma.

**Data-Engineering angle:** CSV and JSON are the two most common raw ingestion formats.
Knowing the standard library tools well removes an entire category of bugs.


In [ ]:

import csv, json, os

os.makedirs("sample_data", exist_ok=True)

# --- Write a CSV file (as if it were "extracted" from a source system) ---
rows = [
    {"id": 1, "name": "Widget A", "price": 9.99, "in_stock": True},
    {"id": 2, "name": "Gadget, Deluxe", "price": 24.50, "in_stock": False},  # note the comma!
    {"id": 3, "name": "Widget B", "price": 14.75, "in_stock": True},
]

csv_path = "sample_data/products.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "name", "price", "in_stock"])
    writer.writeheader()
    writer.writerows(rows)

print("Wrote", csv_path)
with open(csv_path) as f:
    print(f.read())


Wrote sample_data/products.csv
id,name,price,in_stock
1,Widget A,9.99,True
2,"Gadget, Deluxe",24.5,False
3,Widget B,14.75,True



In [ ]:

# --- Read the CSV back — csv.DictReader handles the embedded comma correctly ---
with open(csv_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    loaded = list(reader)

for row in loaded:
    print(row)
    # Note: everything comes back as a STRING — you must cast types yourself (Module 1!)
print("\nType of 'price' column:", type(loaded[0]["price"]))


{'id': '1', 'name': 'Widget A', 'price': '9.99', 'in_stock': 'True'}
{'id': '2', 'name': 'Gadget, Deluxe', 'price': '24.5', 'in_stock': 'False'}
{'id': '3', 'name': 'Widget B', 'price': '14.75', 'in_stock': 'True'}

Type of 'price' column: <class 'str'>


In [ ]:

# --- JSON: nested/semi-structured data (very common for API payloads) ---
order = {
    "order_id": "ORD-2026-001",
    "customer": {"name": "Alice Johnson", "tier": "gold"},
    "items": [
        {"sku": "W-001", "qty": 2, "price": 9.99},
        {"sku": "G-DLX", "qty": 1, "price": 24.50},
    ],
    "total": 44.48,
}

json_path = "sample_data/order.json"
with open(json_path, "w") as f:
    json.dump(order, f, indent=2)          # indent=2 -> human-readable pretty print

with open(json_path) as f:
    loaded_order = json.load(f)

print(f"Customer tier: {loaded_order['customer']['tier']}")
print(f"First item SKU: {loaded_order['items'][0]['sku']}")

# json.dumps / loads (string <-> object) is what you use with APIs, not files
as_string = json.dumps({"a": 1, "b": [1, 2, 3]})
print("\nSerialized:", as_string, "| type:", type(as_string))


Customer tier: gold
First item SKU: W-001

Serialized: {"a": 1, "b": [1, 2, 3]} | type: <class 'str'>



**Try it yourself:** Read `sample_data/products.csv` back in, cast `price` to `float`
and `in_stock` to `bool` (hint: compare the string to `"True"`), and print the total
value of all in-stock products (`sum(price)` for rows where `in_stock` is true).


In [ ]:

# Your code here


# --- Solution ---
# with open(csv_path, newline="", encoding="utf-8") as f:
#     rows = list(csv.DictReader(f))
# total = sum(float(r["price"]) for r in rows if r["in_stock"] == "True")
# print(f"Total in-stock value: {total:.2f}")



## Module 9 — Working with Dates & Times

**Concept:** The `datetime` module is the standard for parsing, formatting, and doing
arithmetic on dates/times. `strptime` **parses** a string into a `datetime`; `strftime`
**formats** a `datetime` back into a string. Be deliberate about time zones — naive
datetimes (no tz) are a classic source of bugs in distributed systems.

**Data-Engineering angle:** Partitioning data by date, computing SLAs ("must land
within 24 hours"), and reconciling timestamps across systems in different time zones
are daily concerns.


In [ ]:

from datetime import datetime, timedelta, timezone

# Parsing a raw timestamp string coming from a source system
raw_ts = "2026-07-08 14:32:11"
dt = datetime.strptime(raw_ts, "%Y-%m-%d %H:%M:%S")
print("Parsed:", dt, "| type:", type(dt).__name__)

# Formatting for a partition path, e.g. s3://bucket/events/year=2026/month=07/day=08/
partition_path = dt.strftime("year=%Y/month=%m/day=%d")
print("Partition path:", partition_path)

# Date arithmetic — SLA check: was this processed within 24 hours of the event?
event_time = dt
processed_time = dt + timedelta(hours=20, minutes=15)
sla_hours = 24
within_sla = (processed_time - event_time) <= timedelta(hours=sla_hours)
print(f"Processing delay: {processed_time - event_time} | Within {sla_hours}h SLA? {within_sla}")


Parsed: 2026-07-08 14:32:11 | type: datetime
Partition path: year=2026/month=07/day=08
Processing delay: 20:15:00 | Within 24h SLA? True


In [ ]:

# Time zones — always be explicit in data engineering (avoid naive datetimes!)
utc_time = datetime(2026, 7, 8, 14, 32, 11, tzinfo=timezone.utc)
ist_offset = timezone(timedelta(hours=5, minutes=30))
ist_time = utc_time.astimezone(ist_offset)

print("UTC:", utc_time)
print("IST:", ist_time)

# Unix epoch timestamps — the universal, timezone-free interchange format
epoch = utc_time.timestamp()
print("Epoch seconds:", epoch)
print("Back to datetime:", datetime.fromtimestamp(epoch, tz=timezone.utc))


UTC: 2026-07-08 14:32:11+00:00
IST: 2026-07-08 20:02:11+05:30
Epoch seconds: 1783521131.0
Back to datetime: 2026-07-08 14:32:11+00:00



**Try it yourself:** Given a list of raw timestamp strings
`["2026-01-15 08:00:00", "2026-01-15 23:59:59", "2026-01-16 00:00:01"]`, parse each one
and print which ones fall on `2026-01-15` vs other days (hint: compare `.date()`).


In [ ]:

# Your code here


# --- Solution ---
# raw_list = ["2026-01-15 08:00:00", "2026-01-15 23:59:59", "2026-01-16 00:00:01"]
# from datetime import date
# target = date(2026, 1, 15)
# for raw in raw_list:
#     dt = datetime.strptime(raw, "%Y-%m-%d %H:%M:%S")
#     print(raw, "-> on target day?", dt.date() == target)



## Module 10 — Object-Oriented Programming for Pipelines

**Concept:** Classes bundle data (attributes) and behavior (methods). Key ideas:
`__init__` (constructor), inheritance (share behavior across related classes),
`@dataclass` (auto-generates boilerplate for data-holding classes), and dunder methods
like `__repr__`/`__eq__` for clean debugging.

**Data-Engineering angle:** Instead of passing loose dicts everywhere, model pipeline
steps as classes: a `DataSource` base class with `Extract`/`Transform`/`Load` subclasses
gives you a consistent, testable, swappable interface — the foundation of frameworks
like Airflow operators or dbt models.


In [ ]:

from dataclasses import dataclass, field
from abc import ABC, abstractmethod

@dataclass
class Record:
    '''@dataclass auto-generates __init__, __repr__, __eq__ for us.'''
    id: int
    amount: float
    tags: list = field(default_factory=list)   # mutable defaults need field(default_factory=...)

r1 = Record(id=1, amount=99.5, tags=["priority"])
r2 = Record(id=1, amount=99.5, tags=["priority"])
print(r1)                    # nice auto-generated repr
print("Equal?", r1 == r2)     # auto-generated __eq__ compares by value


Record(id=1, amount=99.5, tags=['priority'])
Equal? True


In [ ]:

# Abstract base class — defines a contract every pipeline "stage" must follow
class PipelineStage(ABC):
    @abstractmethod
    def run(self, data: list) -> list:
        '''Every stage must implement run().'''

class DeduplicateStage(PipelineStage):
    def run(self, data: list) -> list:
        seen, result = set(), []
        for record in data:
            if record.id not in seen:
                seen.add(record.id)
                result.append(record)
        return result

class FilterPositiveAmountStage(PipelineStage):
    def run(self, data: list) -> list:
        return [r for r in data if r.amount > 0]

# Compose stages into a tiny pipeline
records = [Record(1, 50.0), Record(1, 50.0), Record(2, -10.0), Record(3, 75.0)]
stages = [DeduplicateStage(), FilterPositiveAmountStage()]

for stage in stages:
    records = stage.run(records)
    print(f"After {type(stage).__name__}: {records}")


After DeduplicateStage: [Record(id=1, amount=50.0, tags=[]), Record(id=2, amount=-10.0, tags=[]), Record(id=3, amount=75.0, tags=[])]
After FilterPositiveAmountStage: [Record(id=1, amount=50.0, tags=[]), Record(id=3, amount=75.0, tags=[])]


In [ ]:

# Inheritance + polymorphism — different sources, same interface
class DataSource(ABC):
    @abstractmethod
    def fetch(self) -> list:
        ...

class CSVSource(DataSource):
    def __init__(self, path):
        self.path = path
    def fetch(self) -> list:
        return [{"source": "csv", "path": self.path}]  # simplified for demo

class APISource(DataSource):
    def __init__(self, url):
        self.url = url
    def fetch(self) -> list:
        return [{"source": "api", "url": self.url}]    # simplified for demo

def run_extraction(source: DataSource) -> list:
    '''Doesn't care WHICH source it gets — this is polymorphism in action.'''
    return source.fetch()

for src in [CSVSource("sample_data/products.csv"), APISource("https://api.example.com/orders")]:
    print(run_extraction(src))


[{'source': 'csv', 'path': 'sample_data/products.csv'}]
[{'source': 'api', 'url': 'https://api.example.com/orders'}]



**Try it yourself:** Create a `class Validator` with a method `validate(record: dict) ->
bool` that returns `True` only if `record["amount"] > 0` **and** `"id" in record`. Then
create a subclass `StrictValidator` that also requires `record["amount"] < 10000`.


In [ ]:

# Your code here


# --- Solution ---
# class Validator:
#     def validate(self, record: dict) -> bool:
#         return "id" in record and record.get("amount", -1) > 0
#
# class StrictValidator(Validator):
#     def validate(self, record: dict) -> bool:
#         return super().validate(record) and record["amount"] < 10000
#
# v = StrictValidator()
# print(v.validate({"id": 1, "amount": 500}))
# print(v.validate({"id": 1, "amount": 50000}))



## Module 11 — Modules, Packages & Virtual Environments

**Concept:** A **module** is a single `.py` file; a **package** is a folder of modules
with an `__init__.py`. `import` brings code from one file into another. **Virtual
environments** (`venv`) isolate a project's dependencies from your system Python and
from other projects.

**Data-Engineering angle:** Real pipelines are never one script — you'll organize code
into `extract.py`, `transform.py`, `load.py`, `utils/validators.py`, etc. Virtual envs +
a `requirements.txt` (or `pyproject.toml`) make your pipeline reproducible on another
machine or in CI/CD.


In [ ]:

import os

# Simulate a small package structure for a pipeline project
os.makedirs("etl_project/etl", exist_ok=True)

with open("etl_project/etl/__init__.py", "w") as f:
    f.write("")  # marks 'etl' as a package

with open("etl_project/etl/transform.py", "w") as f:
    f.write(
        'def to_upper(records):\n'
        "    # Uppercase a 'name' field on each record\n"
        '    return [{**r, "name": r["name"].upper()} for r in records]\n'
    )

with open("etl_project/main.py", "w") as f:
    f.write(
        'from etl.transform import to_upper\n\n'
        'data = [{"id": 1, "name": "widget"}]\n'
        'print(to_upper(data))\n'
    )

print("Project structure:")
for root, dirs, files in os.walk("etl_project"):
    level = root.replace("etl_project", "").count(os.sep)
    print("  " * level + os.path.basename(root or "etl_project") + "/")
    for f in files:
        print("  " * (level + 1) + f)


Project structure:
etl_project/
  main.py
  etl/
    __init__.py
    transform.py


In [ ]:

import subprocess, sys

# Run main.py as a real subprocess from inside etl_project/ to prove the import works
result = subprocess.run(
    [sys.executable, "main.py"],
    cwd="etl_project",
    capture_output=True, text=True,
)
print("STDOUT:", result.stdout.strip())
print("STDERR:", result.stderr.strip() or "(none)")


STDOUT: [{'id': 1, 'name': 'WIDGET'}]
STDERR: (none)



**Creating a virtual environment (run these in your terminal, not this notebook):**

```bash
python3 -m venv .venv              # create an isolated environment
source .venv/bin/activate          # activate it (Windows: .venv\Scripts\activate)
pip install pandas sqlalchemy      # installs ONLY into .venv, not system Python
pip freeze > requirements.txt      # snapshot exact versions for reproducibility
pip install -r requirements.txt    # anyone else can now recreate your exact environment
```

**Why this matters:** without isolation, "works on my machine" is a constant risk —
one project needing `pandas==1.5` and another needing `pandas==2.2` will conflict on a
shared system Python install.



## Module 12 — NumPy Fundamentals

**Concept:** NumPy's `ndarray` stores homogeneous numeric data in contiguous memory and
supports **vectorized** operations — applying an operation to an entire array at once,
in fast compiled C code, instead of a slow Python `for` loop.

**Data-Engineering angle:** NumPy is the computational engine under `pandas`. Whenever
you do math over a large numeric column, it's really NumPy doing the work — understanding
it helps you write faster, more memory-efficient transformations.


In [ ]:

import numpy as np

# Vectorized operations vs. Python loops
prices = np.array([19.99, 45.50, 3.25, 120.00, 8.75])
tax_rate = 0.08

# NO explicit loop needed — this applies to the WHOLE array at once
prices_with_tax = prices * (1 + tax_rate)
print("Prices with tax:", np.round(prices_with_tax, 2))

# Boolean masking — a core pattern for filtering data
expensive_mask = prices > 20
print("Expensive mask:", expensive_mask)
print("Expensive items:", prices[expensive_mask])

# Aggregate stats, computed in optimized C code
print(f"\nMean: {prices.mean():.2f} | Std: {prices.std():.2f} | Max: {prices.max()}")


Prices with tax: [ 21.59  49.14   3.51 129.6    9.45]
Expensive mask: [False  True False  True False]
Expensive items: [ 45.5 120. ]

Mean: 39.50 | Std: 42.79 | Max: 120.0


In [ ]:

import time

# Why vectorization matters: speed comparison at scale
n = 2_000_000
python_list = list(range(n))
numpy_array = np.arange(n)

start = time.perf_counter()
python_result = [x * 2 for x in python_list]
python_time = time.perf_counter() - start

start = time.perf_counter()
numpy_result = numpy_array * 2
numpy_time = time.perf_counter() - start

print(f"Pure Python loop: {python_time:.4f}s")
print(f"NumPy vectorized: {numpy_time:.4f}s")
print(f"NumPy is ~{python_time / numpy_time:.0f}x faster on this operation")


Pure Python loop: 0.5745s
NumPy vectorized: 0.1049s
NumPy is ~5x faster on this operation


In [ ]:

# 2D arrays — useful for representing tabular numeric data / feature matrices
sales_matrix = np.array([
    [100, 150, 200],   # store 1: Jan, Feb, Mar
    [80,  95,  110],   # store 2
    [200, 210, 250],   # store 3
])

print("Shape:", sales_matrix.shape)             # (stores, months)
print("Total sales per store:", sales_matrix.sum(axis=1))   # sum across columns
print("Total sales per month:", sales_matrix.sum(axis=0))   # sum across rows
print("Overall total:", sales_matrix.sum())


Shape: (3, 3)
Total sales per store: [450 285 660]
Total sales per month: [380 455 560]
Overall total: 1395



**Try it yourself:** Given `readings = np.array([22.5, 23.1, 45.0, 21.9, -5.0, 22.8])`
(sensor temperature readings, °C), use boolean masking to find all readings that are
**outliers** — outside the range [15, 30] — and print them.


In [ ]:

# Your code here


# --- Solution ---
# readings = np.array([22.5, 23.1, 45.0, 21.9, -5.0, 22.8])
# outliers = readings[(readings < 15) | (readings > 30)]
# print("Outliers:", outliers)



## Module 13 — Pandas: DataFrames

**Concept:** `pandas.DataFrame` is a 2D labeled table (rows + named columns), built on
top of NumPy. It's the workhorse of Python data engineering for anything that fits in
memory: reading/writing CSV/Excel/SQL/Parquet, filtering, joining, grouping, and
reshaping data with a consistent, expressive API.

**Data-Engineering angle:** Most "transform" steps in a small-to-medium ETL job are
written in pandas. Understanding indexing (`.loc`/`.iloc`), vectorized operations, and
`groupby` will cover 80% of daily transformation work.


In [ ]:

import pandas as pd

# Building a DataFrame from a list of dicts (very common — e.g. rows from an API or DB)
data = [
    {"order_id": 1, "customer": "Alice", "region": "US", "amount": 250.0, "status": "COMPLETE"},
    {"order_id": 2, "customer": "Bob",   "region": "EU", "amount": 75.5,  "status": "PENDING"},
    {"order_id": 3, "customer": "Alice", "region": "US", "amount": 900.0, "status": "COMPLETE"},
    {"order_id": 4, "customer": "Carla", "region": "APAC", "amount": 40.0, "status": "FAILED"},
    {"order_id": 5, "customer": "Bob",   "region": "EU", "amount": 310.0, "status": "COMPLETE"},
]
df = pd.DataFrame(data)
df


,order_id,customer,region,amount,status
0,1,Alice,US,250.0,COMPLETE
1,2,Bob,EU,75.5,PENDING
2,3,Alice,US,900.0,COMPLETE
3,4,Carla,APAC,40.0,FAILED
4,5,Bob,EU,310.0,COMPLETE


In [ ]:

# Quick structural checks you should ALWAYS do on new data
print(df.shape)          # (rows, columns)
print(df.dtypes)         # inferred column types
df.info()                 # nulls, dtypes, memory usage in one view


(5, 5)
order_id      int64
customer        str
region          str
amount      float64
status          str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  5 non-null      int64  
 1   customer  5 non-null      str    
 2   region    5 non-null      str    
 3   amount    5 non-null      float64
 4   status    5 non-null      str    
dtypes: float64(1), int64(1), str(3)
memory usage: 402.0 bytes


In [ ]:

# Selecting & filtering — .loc for label-based, boolean masks for filtering
completed_orders = df.loc[df["status"] == "COMPLETE"]
print("Completed orders:\n", completed_orders[["order_id", "customer", "amount"]])

# Multiple conditions: use & / | (NOT `and`/`or`) and wrap each condition in parentheses
big_us_orders = df.loc[(df["region"] == "US") & (df["amount"] > 100)]
print("\nBig US orders:\n", big_us_orders)


Completed orders:
    order_id customer  amount
0         1    Alice   250.0
2         3    Alice   900.0
4         5      Bob   310.0

Big US orders:
    order_id customer region  amount    status
0         1    Alice     US   250.0  COMPLETE
2         3    Alice     US   900.0  COMPLETE


In [ ]:

# groupby — the single most useful pandas operation for aggregation/reporting
revenue_by_customer = df.groupby("customer")["amount"].sum().sort_values(ascending=False)
print("Revenue by customer:\n", revenue_by_customer)

# Multiple aggregations at once with .agg()
summary = df.groupby("region").agg(
    total_amount=("amount", "sum"),
    order_count=("order_id", "count"),
    avg_amount=("amount", "mean"),
)
print("\nRegional summary:\n", summary)


Revenue by customer:
 customer
Alice    1150.0
Bob       385.5
Carla      40.0
Name: amount, dtype: float64

Regional summary:
         total_amount  order_count  avg_amount
region                                       
APAC            40.0            1       40.00
EU             385.5            2      192.75
US            1150.0            2      575.00


In [ ]:

# Adding derived/computed columns — vectorized, no explicit loop needed
df["amount_with_tax"] = df["amount"] * 1.08
df["is_large_order"] = df["amount"] > 200
df


,order_id,customer,region,amount,status,amount_with_tax,is_large_order
0,1,Alice,US,250.0,COMPLETE,270.00,True
1,2,Bob,EU,75.5,PENDING,81.54,False
2,3,Alice,US,900.0,COMPLETE,972.00,True
3,4,Carla,APAC,40.0,FAILED,43.20,False
4,5,Bob,EU,310.0,COMPLETE,334.80,True



**Try it yourself:** Using `df`, compute the total `amount` for orders with
`status == "COMPLETE"`, grouped by `region`. (Hint: filter first, then `groupby`.)


In [ ]:

# Your code here


# --- Solution ---
# result = df[df["status"] == "COMPLETE"].groupby("region")["amount"].sum()
# print(result)



## Module 14 — Data Cleaning & Transformation

**Concept:** Real-world data is dirty: missing values, duplicates, inconsistent
formatting, wrong types, and outliers. Pandas gives you targeted tools: `isna()`,
`fillna()`, `dropna()`, `drop_duplicates()`, `astype()`, `.str` accessor methods, and
`merge()` for combining datasets.

**Data-Engineering angle:** "Garbage in, garbage out." A huge share of an engineer's
time is spent on this exact step — and doing it systematically (not ad hoc) is what
separates reliable pipelines from fragile ones.


In [ ]:

import numpy as np

# A deliberately messy dataset, like something fresh from a source system
messy = pd.DataFrame({
    "id": [1, 2, 2, 3, 4, 5],
    "name": [" Alice ", "Bob", "Bob", "carla", None, "Eve"],
    "email": ["alice@x.com", "bob@x.com", "bob@x.com", "CARLA@X.COM", "dave@x.com", None],
    "age": [29, np.nan, np.nan, 41, 35, 150],   # 150 is an obvious outlier
})
messy


,id,name,email,age
0,1,Alice,alice@x.com,29.0
1,2,Bob,bob@x.com,NaN
2,2,Bob,bob@x.com,NaN
3,3,carla,CARLA@X.COM,41.0
4,4,NaN,dave@x.com,35.0
5,5,Eve,NaN,150.0


In [ ]:

# 1) Detect missing values
print("Missing values per column:\n", messy.isna().sum())

# 2) Handle duplicates — full-row and id-based duplicates
print("\nFull duplicate rows:\n", messy[messy.duplicated()])


Missing values per column:
 id       0
name     1
email    1
age      2
dtype: int64

Full duplicate rows:
    id name      email  age
2   2  Bob  bob@x.com  NaN


In [ ]:

# 3) Clean step by step, building a pipeline of transformations
clean = messy.copy()

# Deduplicate on the natural key
clean = clean.drop_duplicates(subset="id", keep="first")

# Normalize text columns: strip whitespace, standardize case
clean["name"] = clean["name"].str.strip().str.title()
clean["email"] = clean["email"].str.lower()

# Fill missing numeric values with a sensible default (median is robust to outliers)
clean["age"] = clean["age"].fillna(clean["age"].median())

# Cap unrealistic outliers instead of dropping the whole row
clean["age"] = clean["age"].clip(upper=100)

# Drop rows still missing a critical field (email is required)
clean = clean.dropna(subset=["email"])

clean


,id,name,email,age
0,1,Alice,alice@x.com,29.0
1,2,Bob,bob@x.com,38.0
3,3,Carla,carla@x.com,41.0
4,4,NaN,dave@x.com,35.0


In [ ]:

# Merging two datasets — the pandas equivalent of a SQL JOIN
customers = pd.DataFrame({
    "customer": ["Alice", "Bob", "Carla"],
    "tier": ["gold", "silver", "bronze"],
})

merged = df.merge(customers, on="customer", how="left")   # left join keeps all orders
print(merged[["order_id", "customer", "amount", "tier"]])


   order_id customer  amount    tier
0         1    Alice   250.0    gold
1         2      Bob    75.5  silver
2         3    Alice   900.0    gold
3         4    Carla    40.0  bronze
4         5      Bob   310.0  silver



**Try it yourself:** Given `clean` from above, create a new column `email_domain` that
extracts everything after `@` in the `email` column (hint: `.str.split("@").str[1]`).


In [ ]:

# Your code here


# --- Solution ---
# clean["email_domain"] = clean["email"].str.split("@").str[1]
# print(clean[["email", "email_domain"]])



## Module 15 — Working with Databases (SQLite & SQLAlchemy)

**Concept:** Python talks to databases through **DB-API** drivers (`sqlite3` is built
in; `psycopg2`/`pg8000` for Postgres, etc.). **SQLAlchemy** sits on top as a toolkit
that gives you a consistent interface across database engines, connection pooling, and
(optionally) an ORM. Always use **parameterized queries** — never string-format raw SQL
with user/data values, that's how SQL injection happens.

**Data-Engineering angle:** Loading transformed data into a warehouse/database, or
pulling data out for transformation, is a core "L" (Load) and "E" (Extract) step in ETL.


In [ ]:

import sqlite3

# Connect to a local SQLite file (creates it if it doesn't exist)
conn = sqlite3.connect("sample_data/warehouse.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS orders")
create_sql = (
    "CREATE TABLE orders ("
    "    order_id INTEGER PRIMARY KEY,"
    "    customer TEXT NOT NULL,"
    "    amount REAL NOT NULL,"
    "    status TEXT"
    ")"
)
cursor.execute(create_sql)
conn.commit()
print("Table created.")


Table created.


In [ ]:

# Parameterized INSERT — the "?" placeholders prevent SQL injection
records_to_insert = [
    (1, "Alice", 250.0, "COMPLETE"),
    (2, "Bob", 75.5, "PENDING"),
    (3, "Alice", 900.0, "COMPLETE"),
]

cursor.executemany(
    "INSERT INTO orders (order_id, customer, amount, status) VALUES (?, ?, ?, ?)",
    records_to_insert,
)
conn.commit()
print(f"Inserted {cursor.rowcount if cursor.rowcount != -1 else len(records_to_insert)} rows.")

# NEVER do this (vulnerable to SQL injection):
# cursor.execute(f"INSERT INTO orders VALUES ({order_id}, '{customer}', ...)")


Inserted 3 rows.


In [ ]:

# Querying — fetchall() / fetchone(), and reading results straight into pandas
cursor.execute("SELECT * FROM orders WHERE amount > ?", (100,))
for row in cursor.fetchall():
    print(row)

print()
# pandas makes SQL -> DataFrame trivial, which is extremely common in data engineering
df_from_db = pd.read_sql_query("SELECT customer, SUM(amount) as total FROM orders GROUP BY customer", conn)
print(df_from_db)

conn.close()


(1, 'Alice', 250.0, 'COMPLETE')
(3, 'Alice', 900.0, 'COMPLETE')

  customer   total
0    Alice  1150.0
1      Bob    75.5


In [ ]:

# SQLAlchemy — a higher-level, engine-agnostic interface (same code works for
# SQLite, Postgres, MySQL, etc. by just changing the connection string)
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///sample_data/warehouse.db")

with engine.connect() as connection:
    result = connection.execute(text("SELECT customer, amount FROM orders WHERE status = :status"),
                                 {"status": "COMPLETE"})
    for row in result:
        print(row)

# Loading a DataFrame straight into a table — the "Load" step of ETL in one line
new_data = pd.DataFrame([{"order_id": 4, "customer": "Carla", "amount": 40.0, "status": "FAILED"}])
new_data.to_sql("orders", engine, if_exists="append", index=False)
print("\nLoaded new_data into 'orders' table.")

final = pd.read_sql_table("orders", engine)
print(final)


('Alice', 250.0)
('Alice', 900.0)

Loaded new_data into 'orders' table.
   order_id customer  amount    status
0         1    Alice   250.0  COMPLETE
1         2      Bob    75.5   PENDING
2         3    Alice   900.0  COMPLETE
3         4    Carla    40.0    FAILED



**Try it yourself:** Using the `engine` above, write a query with SQLAlchemy that
computes the count of orders per `status`. (Hint: `SELECT status, COUNT(*) FROM orders
GROUP BY status`.)


In [ ]:

# Your code here


# --- Solution ---
# with engine.connect() as connection:
#     result = connection.execute(text("SELECT status, COUNT(*) as cnt FROM orders GROUP BY status"))
#     for row in result:
#         print(row)



## Module 16 — Consuming APIs & Working with JSON

**Concept:** The `requests` library is the standard for calling HTTP APIs. Responses
are almost always JSON, which maps naturally to Python dicts/lists. Real-world API
work also means handling pagination, rate limits, retries, and timeouts gracefully.

**Data-Engineering angle:** A large share of "Extract" logic pulls data from REST APIs
(SaaS tools, internal microservices, public data sources). Doing this robustly — with
retries and backoff — is what keeps an ingestion job from failing at 3 AM over a
transient network blip.


In [ ]:

import requests

# A public test API — https://jsonplaceholder.typicode.com is a free fake REST API for testing
try:
    response = requests.get("https://jsonplaceholder.typicode.com/users/1", timeout=5)
    response.raise_for_status()          # raises an exception for 4xx/5xx responses
    user = response.json()               # parse JSON body straight into a dict
    print(f"Status code: {response.status_code}")
    print(f"User: {user['name']} <{user['email']}>")
    print(f"Company: {user['company']['name']}")
except requests.exceptions.RequestException as e:
    print(f"API call failed (this is expected if you're offline/sandboxed): {e}")


API call failed (this is expected if you're offline/sandboxed): 403 Client Error: Forbidden for url: https://jsonplaceholder.typicode.com/users/1


In [ ]:

# Handling pagination — a very common real-world pattern
try:
    all_posts = []
    page = 1
    while True:
        resp = requests.get(
            "https://jsonplaceholder.typicode.com/posts",
            params={"_page": page, "_limit": 20},
            timeout=5,
        )
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        all_posts.extend(batch)
        if page >= 3:   # safety cap for this demo
            break
        page += 1

    print(f"Fetched {len(all_posts)} posts across {page} pages")
    print("Sample post:", all_posts[0]["title"][:50])
except requests.exceptions.RequestException as e:
    print(f"API call failed (expected if offline): {e}")


API call failed (expected if offline): 403 Client Error: Forbidden for url: https://jsonplaceholder.typicode.com/posts?_page=1&_limit=20


In [ ]:

import time

# Retry with exponential backoff — essential resilience pattern for flaky APIs
def fetch_with_retry(url, max_retries=3, backoff_base=1.0, **kwargs):
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, timeout=5, **kwargs)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as e:
            wait = backoff_base * (2 ** (attempt - 1))
            print(f"Attempt {attempt} failed ({e}); retrying in {wait:.1f}s..." if attempt < max_retries
                  else f"Attempt {attempt} failed ({e}); giving up.")
            if attempt < max_retries:
                time.sleep(wait)
    return None

result = fetch_with_retry("https://jsonplaceholder.typicode.com/users/2")
if result:
    print("Success:", result["name"])


Attempt 1 failed (403 Client Error: Forbidden for url: https://jsonplaceholder.typicode.com/users/2); retrying in 1.0s...


Attempt 2 failed (403 Client Error: Forbidden for url: https://jsonplaceholder.typicode.com/users/2); retrying in 2.0s...


Attempt 3 failed (403 Client Error: Forbidden for url: https://jsonplaceholder.typicode.com/users/2); giving up.



**Try it yourself:** Fetch `https://jsonplaceholder.typicode.com/todos?userId=1` and
count how many todos have `"completed": True` vs `False`.


In [ ]:

# Your code here


# --- Solution ---
# resp = requests.get("https://jsonplaceholder.typicode.com/todos", params={"userId": 1}, timeout=5)
# todos = resp.json()
# from collections import Counter
# print(Counter(t["completed"] for t in todos))



## Module 17 — Columnar Formats: Parquet (PyArrow)

**Concept:** CSV/JSON are row-oriented, human-readable, but slow and storage-heavy at
scale. **Parquet** is a columnar, compressed, binary format — reading only the columns
you need is much faster, and file sizes shrink dramatically. It also stores schema/types
natively (no more "everything is a string" problem from CSV).

**Data-Engineering angle:** Parquet is the de facto standard for data lakes (S3, ADLS,
GCS) and is what tools like Spark, Athena, BigQuery, and Snowflake read natively.
Knowing how to read/write it — and why it beats CSV at scale — is essential.


In [ ]:

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

# Generate a moderately sized sample dataset
import numpy as np
n = 50_000
big_df = pd.DataFrame({
    "id": np.arange(n),
    "region": np.random.choice(["US", "EU", "APAC"], size=n),
    "amount": np.round(np.random.uniform(10, 1000, size=n), 2),
    "status": np.random.choice(["COMPLETE", "PENDING", "FAILED"], size=n, p=[0.7, 0.2, 0.1]),
})

csv_path = "sample_data/big_orders.csv"
parquet_path = "sample_data/big_orders.parquet"

big_df.to_csv(csv_path, index=False)
big_df.to_parquet(parquet_path, index=False)   # pandas uses pyarrow under the hood

csv_size = os.path.getsize(csv_path)
parquet_size = os.path.getsize(parquet_path)
print(f"CSV size:     {csv_size:,} bytes")
print(f"Parquet size: {parquet_size:,} bytes")
print(f"Parquet is ~{csv_size / parquet_size:.1f}x smaller")


CSV size:     1,242,552 bytes
Parquet size: 612,716 bytes
Parquet is ~2.0x smaller


In [ ]:

import time

# Speed comparison: reading only 1 column out of 4
start = time.perf_counter()
_ = pd.read_csv(csv_path, usecols=["amount"])
csv_time = time.perf_counter() - start

start = time.perf_counter()
_ = pd.read_parquet(parquet_path, columns=["amount"])
parquet_time = time.perf_counter() - start

print(f"CSV read (1 column):     {csv_time*1000:.1f} ms")
print(f"Parquet read (1 column): {parquet_time*1000:.1f} ms")
print("-> Parquet's columnar layout means it doesn't have to scan unused columns at all.")


CSV read (1 column):     43.1 ms
Parquet read (1 column): 24.6 ms
-> Parquet's columnar layout means it doesn't have to scan unused columns at all.


In [ ]:

# Schema is preserved automatically — no manual type casting needed after reading!
reloaded = pd.read_parquet(parquet_path)
print(reloaded.dtypes)

# PyArrow also lets you inspect metadata without loading the full file into memory
parquet_file = pq.ParquetFile(parquet_path)
print(f"\nRow count: {parquet_file.metadata.num_rows:,}")
print(f"Columns: {parquet_file.schema.names}")


id          int64
region        str
amount    float64
status        str
dtype: object

Row count: 50,000
Columns: ['id', 'region', 'amount', 'status']



**Try it yourself:** Read only the `region` and `status` columns from
`sample_data/big_orders.parquet`, then print the count of rows per `(region, status)`
combination using `groupby`.


In [ ]:

# Your code here


# --- Solution ---
# subset = pd.read_parquet(parquet_path, columns=["region", "status"])
# print(subset.groupby(["region", "status"]).size())



## Module 18 — Concurrency & Parallelism

**Concept:** Python has three main approaches to doing multiple things "at once":
- **Threading** — good for I/O-bound work (API calls, file/network reads) where threads
  spend most of their time *waiting*. Limited by the GIL for CPU work.
- **Multiprocessing** — good for CPU-bound work (heavy computation) since each process
  gets its own Python interpreter and bypasses the GIL.
- **asyncio** — good for very high-volume I/O-bound work (thousands of concurrent
  network calls) using a single thread and cooperative scheduling.

**Data-Engineering angle:** Fetching data from 50 API endpoints, or transforming 20
large files, sequentially wastes enormous time. Choosing the right concurrency tool for
the bottleneck (I/O vs CPU) is a key performance skill.


In [ ]:

import time
from concurrent.futures import ThreadPoolExecutor

# Simulate a slow I/O operation, e.g. an API call or a network file read
def slow_io_task(task_id):
    time.sleep(0.3)   # pretend this is waiting on the network
    return f"task-{task_id} done"

task_ids = list(range(8))

# Sequential (slow) — waits for each I/O operation one after another
start = time.perf_counter()
sequential_results = [slow_io_task(i) for i in task_ids]
sequential_time = time.perf_counter() - start

# Concurrent with threads — I/O-bound, so threads overlap their waiting time
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as executor:
    threaded_results = list(executor.map(slow_io_task, task_ids))
threaded_time = time.perf_counter() - start

print(f"Sequential: {sequential_time:.2f}s")
print(f"Threaded:   {threaded_time:.2f}s  (~{sequential_time/threaded_time:.1f}x faster)")


Sequential: 2.40s
Threaded:   0.31s  (~7.9x faster)


In [ ]:

from concurrent.futures import ProcessPoolExecutor
import math

# Simulate a CPU-bound task — heavy computation benefits from separate PROCESSES, not threads
def cpu_bound_task(n):
    return sum(math.sqrt(i) for i in range(n))

numbers = [3_000_000] * 4

start = time.perf_counter()
sequential = [cpu_bound_task(n) for n in numbers]
seq_time = time.perf_counter() - start
print(f"Sequential CPU-bound: {seq_time:.2f}s")

# NOTE: ProcessPoolExecutor needs a real __main__ guard in a .py script;
# in notebooks it generally works directly, but behavior can vary by OS.
# The pattern to try in your own environment:
print("from concurrent.futures import ProcessPoolExecutor")
print("with ProcessPoolExecutor(max_workers=4) as executor:")
print("    results = list(executor.map(cpu_bound_task, numbers))")


Sequential CPU-bound: 1.08s
from concurrent.futures import ProcessPoolExecutor
with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(cpu_bound_task, numbers))



**Try it yourself:** Using `ThreadPoolExecutor`, fetch 5 different JSONPlaceholder user
records (`/users/1` through `/users/5`) concurrently and print each user's name. Compare
timing against fetching them one at a time in a loop.


In [ ]:

# Your code here


# --- Solution ---
# import requests
# from concurrent.futures import ThreadPoolExecutor
#
# def fetch_user(user_id):
#     resp = requests.get(f"https://jsonplaceholder.typicode.com/users/{user_id}", timeout=5)
#     return resp.json()["name"]
#
# with ThreadPoolExecutor(max_workers=5) as executor:
#     names = list(executor.map(fetch_user, range(1, 6)))
# print(names)



## Module 19 — Config, Environment & Secrets

**Concept:** Never hard-code database passwords, API keys, or environment-specific
values (dev vs prod file paths) directly in your code. Use environment variables
(`os.environ`), `.env` files (loaded via `python-dotenv`), or a config file
(YAML/JSON/TOML) read at startup.

**Data-Engineering angle:** A pipeline that hard-codes `password="admin123"` is a
security incident waiting to happen, and one that hard-codes `bucket="dev-bucket"` will
silently write to the wrong place the day it's promoted to production.


In [ ]:

import os

# Reading configuration from environment variables, with sensible defaults
DB_HOST = os.environ.get("DB_HOST", "localhost")
DB_PORT = int(os.environ.get("DB_PORT", "5432"))
ENVIRONMENT = os.environ.get("APP_ENV", "development")

print(f"Connecting to {DB_HOST}:{DB_PORT} in '{ENVIRONMENT}' mode")

# Simulate setting an env var (in real life this comes from your shell, Docker, or CI/CD)
os.environ["APP_ENV"] = "production"
print("After override:", os.environ.get("APP_ENV"))


Connecting to localhost:5432 in 'development' mode
After override: production


In [ ]:

# A minimal config pattern using a dataclass — type-safe, testable, and central
from dataclasses import dataclass

@dataclass(frozen=True)
class PipelineConfig:
    db_host: str
    db_port: int
    batch_size: int = 1000
    max_retries: int = 3

def load_config() -> PipelineConfig:
    return PipelineConfig(
        db_host=os.environ.get("DB_HOST", "localhost"),
        db_port=int(os.environ.get("DB_PORT", "5432")),
        batch_size=int(os.environ.get("BATCH_SIZE", "1000")),
    )

config = load_config()
print(config)


PipelineConfig(db_host='localhost', db_port=5432, batch_size=1000, max_retries=3)



**Secrets — the golden rules:**
1. Never commit secrets to version control (use `.gitignore` for `.env` files).
2. Use a secrets manager in production (AWS Secrets Manager, HashiCorp Vault, GCP
   Secret Manager) rather than plain environment variables where possible.
3. Rotate credentials periodically and scope them to the minimum required permissions.

**A `.env` file (loaded with `python-dotenv`) looks like this:**

```
DB_HOST=prod-db.internal
DB_PASSWORD=use-a-secrets-manager-instead-of-this-in-real-prod
API_KEY=sk_live_xxx
```

```python
from dotenv import load_dotenv
load_dotenv()   # populates os.environ from the .env file
```



## Module 20 — Capstone: Building an End-to-End Mini ETL Pipeline

**Concept:** Let's tie everything together — data structures, functions, exception
handling, file I/O, pandas, and databases — into one small but *realistic* pipeline:

**Extract** → read raw CSV → **Transform** → clean, validate, enrich → **Load** → write
to SQLite, with logging and a dead-letter path for bad records throughout.


In [ ]:

import logging
import pandas as pd
import sqlite3
from dataclasses import dataclass

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("capstone_etl")

@dataclass
class ETLResult:
    rows_extracted: int
    rows_loaded: int
    rows_rejected: int


In [ ]:

# --- EXTRACT ---
def extract(csv_path: str) -> pd.DataFrame:
    logger.info(f"Extracting from {csv_path}")
    df = pd.read_csv(csv_path)
    logger.info(f"Extracted {len(df)} raw rows")
    return df

# Create a fresh messy source file for this demo
raw_source = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 5],
    "customer": [" Alice ", "bob", "Carla", None, "Eve", "Eve"],
    "amount": ["250.00", "75.5", "N/A", "900", "-40", "-40"],
    "order_date": ["2026-07-01", "2026-07-02", "2026-07-02", "2026-07-03", "2026-07-04", "2026-07-04"],
})
raw_source.to_csv("sample_data/capstone_orders.csv", index=False)

extracted = extract("sample_data/capstone_orders.csv")
extracted


11:30:29 | INFO     | Extracting from sample_data/capstone_orders.csv


11:30:29 | INFO     | Extracted 6 raw rows


,order_id,customer,amount,order_date
0,1,Alice,250.0,2026-07-01
1,2,bob,75.5,2026-07-02
2,3,Carla,NaN,2026-07-02
3,4,NaN,900.0,2026-07-03
4,5,Eve,-40.0,2026-07-04
5,5,Eve,-40.0,2026-07-04


In [ ]:

# --- TRANSFORM ---
def transform(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    logger.info("Transforming data")
    df = df.copy()

    # Deduplicate on natural key
    df = df.drop_duplicates(subset="order_id", keep="first")

    # Normalize text
    df["customer"] = df["customer"].str.strip().str.title()

    # Coerce amount to numeric; invalid values become NaN (they'll be rejected below)
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    # Parse dates
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

    # Split into clean vs rejected based on business rules
    is_valid = df["customer"].notna() & df["amount"].notna() & (df["amount"] > 0) & df["order_date"].notna()
    clean_df = df[is_valid].reset_index(drop=True)
    rejected_df = df[~is_valid].reset_index(drop=True)

    logger.info(f"Transform complete: {len(clean_df)} valid, {len(rejected_df)} rejected")
    return clean_df, rejected_df

clean_df, rejected_df = transform(extracted)
print("Clean:\n", clean_df)
print("\nRejected:\n", rejected_df)


11:30:29 | INFO     | Transforming data


11:30:29 | INFO     | Transform complete: 2 valid, 3 rejected


Clean:
    order_id customer  amount order_date
0         1    Alice   250.0 2026-07-01
1         2      Bob    75.5 2026-07-02

Rejected:
    order_id customer  amount order_date
0         3    Carla     NaN 2026-07-02
1         4      NaN   900.0 2026-07-03
2         5      Eve   -40.0 2026-07-04


In [ ]:

# --- LOAD ---
def load(df: pd.DataFrame, db_path: str, table_name: str) -> int:
    logger.info(f"Loading {len(df)} rows into {table_name}")
    conn = sqlite3.connect(db_path)
    try:
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        conn.commit()
    finally:
        conn.close()
    return len(df)

rows_loaded = load(clean_df, "sample_data/warehouse.db", "clean_orders")
print(f"Loaded {rows_loaded} rows.")


11:30:29 | INFO     | Loading 2 rows into clean_orders


Loaded 2 rows.


In [ ]:

# --- ORCHESTRATE: run the full pipeline end-to-end with a summary report ---
def run_pipeline(csv_path: str, db_path: str) -> ETLResult:
    try:
        raw = extract(csv_path)
        clean, rejected = transform(raw)

        if not rejected.empty:
            rejected.to_csv("sample_data/capstone_rejected.csv", index=False)
            logger.warning(f"{len(rejected)} rows written to dead-letter file")

        loaded_count = load(clean, db_path, "clean_orders")

        return ETLResult(
            rows_extracted=len(raw),
            rows_loaded=loaded_count,
            rows_rejected=len(rejected),
        )
    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        raise

result = run_pipeline("sample_data/capstone_orders.csv", "sample_data/warehouse.db")
print("\n=== ETL SUMMARY ===")
print(result)


11:30:29 | INFO     | Extracting from sample_data/capstone_orders.csv


11:30:29 | INFO     | Extracted 6 raw rows


11:30:29 | INFO     | Transforming data


11:30:29 | INFO     | Transform complete: 2 valid, 3 rejected


11:30:29 | WARNING  | 3 rows written to dead-letter file


11:30:29 | INFO     | Loading 2 rows into clean_orders



=== ETL SUMMARY ===
ETLResult(rows_extracted=6, rows_loaded=2, rows_rejected=3)


In [ ]:

# Verify the load — read it straight back from the database
conn = sqlite3.connect("sample_data/warehouse.db")
final_check = pd.read_sql_query("SELECT * FROM clean_orders", conn)
conn.close()
final_check


,order_id,customer,amount,order_date
0,1,Alice,250.0,2026-07-01 00:00:00
1,2,Bob,75.5,2026-07-02 00:00:00



**Try it yourself (final challenge):** Extend `run_pipeline` to also compute and print
total revenue (`sum(amount)`) per `customer` from `clean_df` before loading, and add a
new column `revenue_tier` to the DataFrame: `"high"` if `amount > 500`, else `"low"`,
before it gets loaded into the database.


In [ ]:

# Your code here


# --- Solution ---
# clean_df["revenue_tier"] = clean_df["amount"].apply(lambda a: "high" if a > 500 else "low")
# print(clean_df.groupby("customer")["amount"].sum())
# load(clean_df, "sample_data/warehouse.db", "clean_orders")



---
## 🎓 Course Complete!

You've covered the full arc of Python for Data Engineering:

**Foundations** (variables, control flow, functions, data structures, strings/regex)
→ **Efficient processing** (comprehensions, generators) → **Robustness** (exceptions,
logging) → **I/O** (files, dates, OOP, packaging) → **The data stack** (NumPy, pandas,
SQL/SQLAlchemy, REST APIs, Parquet) → **Performance** (concurrency) → **Production
readiness** (config/secrets) → **A real pipeline** (the capstone).

**Suggested next steps:**
- Rebuild the capstone pipeline against a real API and a real Postgres/MySQL database.
- Explore orchestration tools (Airflow, Dagster, Prefect) to schedule this pipeline.
- Learn dbt for SQL-based transformation layers on top of a warehouse.
- Practice writing unit tests (`pytest`) for each Extract/Transform/Load function.
